# Project · Find groups without labels: k-means, Gaussian mixtures and EM

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app). No labels this time. You'll write k-means and a full-covariance Gaussian mixture fitted by EM, both from scratch. Run them on the iris flowers (hiding the species) and check whether they rediscover the species. Then see where k-means' 'round clusters' assumption fails, and pick the number of clusters with BIC. Uses Murphy ch. 3, 8 and 21.

Work through the milestones in order; each builds on the previous cell's variables. Solutions are hidden: double-click a solution cell's title to reveal it.

## Milestone 1 · k-means from scratch

Standardise the 4 iris measurements (ignore the species). Implement k-means: assign each point to its nearest centre, then move each centre to the mean of its points. Repeat until nothing changes. Use 10 random starts and keep the lowest inertia (sum of squared distances).

In [ ]:
# Your turn
import numpy as np
from sklearn.datasets import load_iris
iris = load_iris(); X = (iris.data - iris.data.mean(0)) / iris.data.std(0); species = iris.target
def kmeans(X, k, seed):
    r = np.random.default_rng(seed); C = X[r.choice(len(X), k, replace=False)]
    while True:
        lab = ...  # TODO: nearest centre
        newC = np.array([X[lab == j].mean(0) if np.any(lab == j) else C[j] for j in range(k)])
        if np.allclose(newC, C): break
        C = newC
    return lab, C, ((X - C[lab]) ** 2).sum()
runs = [kmeans(X, 3, s) for s in range(10)]
lab, C, inertia = min(runs, key=lambda t: t[2])
print("inertia per start:", np.round([t[2] for t in runs], 1))
print(f"best inertia {inertia:.1f}, cluster sizes {np.bincount(lab)}")

In [ ]:
# @title Solution 1 (double-click to show the code)
import numpy as np
from sklearn.datasets import load_iris
iris = load_iris(); X = (iris.data - iris.data.mean(0)) / iris.data.std(0); species = iris.target
def kmeans(X, k, seed):
    r = np.random.default_rng(seed); C = X[r.choice(len(X), k, replace=False)]
    while True:
        lab = np.argmin(((X[:, None, :] - C[None]) ** 2).sum(2), 1)  # nearest centre
        newC = np.array([X[lab == j].mean(0) if np.any(lab == j) else C[j] for j in range(k)])
        if np.allclose(newC, C): break
        C = newC
    return lab, C, ((X - C[lab]) ** 2).sum()
runs = [kmeans(X, 3, s) for s in range(10)]
lab, C, inertia = min(runs, key=lambda t: t[2])
print("inertia per start:", np.round([t[2] for t in runs], 1))
print(f"best inertia {inertia:.1f}, cluster sizes {np.bincount(lab)}")

<details><summary><b>Answer and explanation</b></summary>

The best inertia is 139.8 with clusters of 47, 50 and 53. Two of the ten starts got stuck at 191, a clearly worse local minimum. k-means only ever lowers the inertia (each step does), so it always stops, but where depends on the start. That is why you restart and keep the best (Murphy §21.3).

Expected output:
```
inertia per start: [140.  140.9 191.  140.9 139.8 140.1 140.9 140.  191.  140. ]
best inertia 139.8, cluster sizes [47 50 53]
```

</details>

## Milestone 2 · The elbow

Run k-means for k = 1 … 8 (best of 10 starts each) and print the inertia. Where does adding a cluster stop paying off?

In [ ]:
# Your turn
for k in range(1, 9):
    best = ...  # TODO: best of 10 restarts
    print(f"k = {k}: inertia {best:7.1f}")

In [ ]:
# @title Solution 2 (double-click to show the code)
for k in range(1, 9):
    best = min(kmeans(X, k, s)[2] for s in range(10))  # best of 10 restarts
    print(f"k = {k}: inertia {best:7.1f}")

<details><summary><b>Answer and explanation</b></summary>

Inertia: 600 → 222 → 140 → 114 → 91 …. The big drops are k = 1→2 (−378) and 2→3 (−83). After that each extra cluster buys less than 30. The elbow is at 2–3: one very distinct species (setosa) gives the first drop, and versicolor and virginica overlap, giving the smaller second one. Inertia always falls as k grows (k = n gives 0), so it can't pick k on its own.

Expected output:
```
k = 1: inertia   600.0
k = 2: inertia   222.4
k = 3: inertia   139.8
k = 4: inertia   114.1
k = 5: inertia    91.0
k = 6: inertia    81.5
k = 7: inertia    72.5
k = 8: inertia    64.3
```

</details>

## Milestone 3 · Did it rediscover the species?

Cross-tabulate the k = 3 clusters against the hidden species. What is the purity (the fraction of points that belong to their cluster's majority species)?

In [ ]:
# Your turn
tab = np.zeros((3, 3), int)
for c, s in zip(lab, species): tab[c, s] += 1
print("rows = clusters, columns =", [str(n) for n in iris.target_names]); print(tab)
...  # TODO: majority species per cluster

In [ ]:
# @title Solution 3 (double-click to show the code)
tab = np.zeros((3, 3), int)
for c, s in zip(lab, species): tab[c, s] += 1
print("rows = clusters, columns =", [str(n) for n in iris.target_names]); print(tab)
print(f"purity {tab.max(1).sum() / len(X):.3f}")  # majority species per cluster

<details><summary><b>Answer and explanation</b></summary>

Purity 0.833. Setosa is found perfectly (one cluster holds all 50). Versicolor and virginica get mixed: 11 versicolor land in the 'virginica' cluster and 14 virginica in the 'versicolor' one. Those two species overlap and their clouds are elongated, while k-means draws boundaries as if every cluster were a round ball of the same size.

Expected output:
```
rows = clusters, columns = ['setosa', 'versicolor', 'virginica']
[[ 0 11 36]
 [50  0  0]
 [ 0 39 14]]
purity 0.833
```

</details>

## Milestone 4 · A Gaussian mixture fitted by EM

Fit a 3-component Gaussian mixture with full covariances. E-step: responsibilities r_nk ∝ π_k N(x_n | μ_k, Σ_k). M-step: weighted means, covariances and mixing weights. Start from the k-means solution and print the log-likelihood every few iterations. It must never decrease. Compute purity again.

In [ ]:
# Your turn
def logN(X, m, S):
    D = X.shape[1]; L = np.linalg.cholesky(S); z = np.linalg.solve(L, (X - m).T)
    return -0.5 * (z ** 2).sum(0) - np.log(np.diag(L)).sum() - 0.5 * D * np.log(2 * np.pi)
def gmm(X, k, init_lab, iters=100):
    pi = np.bincount(init_lab, minlength=k) / len(X)
    mu = np.array([X[init_lab == j].mean(0) for j in range(k)])
    Sg = np.array([np.cov(X[init_lab == j].T) + 1e-4 * np.eye(X.shape[1]) for j in range(k)])
    hist = []
    for it in range(iters):
        lp = np.stack([np.log(pi[j]) + logN(X, mu[j], Sg[j]) for j in range(k)], 1)
        mx = lp.max(1, keepdims=True); ll = (mx[:, 0] + np.log(np.exp(lp - mx).sum(1))).sum(); hist.append(ll)
        R = np.exp(lp - mx); R /= ...  # TODO: E-step: responsibilities
        Nk = R.sum(0); pi = Nk / len(X); mu = ...  # TODO: M-step
        Sg = np.array([((R[:, j, None] * (X - mu[j])).T @ (X - mu[j])) / Nk[j] + 1e-4 * np.eye(X.shape[1]) for j in range(k)])
    return R, mu, Sg, pi, hist
R, mu, Sg, pi, hist = gmm(X, 3, lab)
print("log-likelihood at iterations 0, 1, 2, 5, 10, 20, 50, 99:", [round(float(hist[i]), 1) for i in (0, 1, 2, 5, 10, 20, 50, 99)])
print("never decreased:", bool(np.all(np.diff(hist) > -1e-8)))
g = R.argmax(1); tab = np.zeros((3, 3), int)
for c, s in zip(g, species): tab[c, s] += 1
print(tab); print(f"GMM purity {tab.max(1).sum() / len(X):.3f}")

In [ ]:
# @title Solution 4 (double-click to show the code)
def logN(X, m, S):
    D = X.shape[1]; L = np.linalg.cholesky(S); z = np.linalg.solve(L, (X - m).T)
    return -0.5 * (z ** 2).sum(0) - np.log(np.diag(L)).sum() - 0.5 * D * np.log(2 * np.pi)
def gmm(X, k, init_lab, iters=100):
    pi = np.bincount(init_lab, minlength=k) / len(X)
    mu = np.array([X[init_lab == j].mean(0) for j in range(k)])
    Sg = np.array([np.cov(X[init_lab == j].T) + 1e-4 * np.eye(X.shape[1]) for j in range(k)])
    hist = []
    for it in range(iters):
        lp = np.stack([np.log(pi[j]) + logN(X, mu[j], Sg[j]) for j in range(k)], 1)
        mx = lp.max(1, keepdims=True); ll = (mx[:, 0] + np.log(np.exp(lp - mx).sum(1))).sum(); hist.append(ll)
        R = np.exp(lp - mx); R /= R.sum(1, keepdims=True)  # E-step: responsibilities
        Nk = R.sum(0); pi = Nk / len(X); mu = (R.T @ X) / Nk[:, None]  # M-step
        Sg = np.array([((R[:, j, None] * (X - mu[j])).T @ (X - mu[j])) / Nk[j] + 1e-4 * np.eye(X.shape[1]) for j in range(k)])
    return R, mu, Sg, pi, hist
R, mu, Sg, pi, hist = gmm(X, 3, lab)
print("log-likelihood at iterations 0, 1, 2, 5, 10, 20, 50, 99:", [round(float(hist[i]), 1) for i in (0, 1, 2, 5, 10, 20, 50, 99)])
print("never decreased:", bool(np.all(np.diff(hist) > -1e-8)))
g = R.argmax(1); tab = np.zeros((3, 3), int)
for c, s in zip(g, species): tab[c, s] += 1
print(tab); print(f"GMM purity {tab.max(1).sum() / len(X):.3f}")

<details><summary><b>Answer and explanation</b></summary>

The log-likelihood climbs from −321.0 to −290.5 and never decreases: that's EM's guarantee, since each E-step and each M-step can only raise a lower bound that touches the log-likelihood (Murphy §8.7.2). Purity jumps to 0.967, with only 5 mistakes. With full covariances each species can be a tilted, stretched ellipse, which is what versicolor and virginica actually look like.

Expected output:
```
log-likelihood at iterations 0, 1, 2, 5, 10, 20, 50, 99: [-321.0, -316.8, -315.2, -306.3, -294.8, -290.5, -290.5, -290.5]
never decreased: True
[[ 0  5 50]
 [50  0  0]
 [ 0 45  0]]
GMM purity 0.967
```

</details>

## Milestone 5 · Where round clusters fail

Make two long, thin, parallel clusters (Gaussians with covariance [[6, 0], [0, 0.05]], centres 1.2 apart vertically, 300 points each). Compare k-means' accuracy with the GMM's, taking the better of the two ways to match cluster ids to true groups.

In [ ]:
# Your turn
r = np.random.default_rng(3)
Z = np.vstack([r.multivariate_normal([0, 0], [[6, 0], [0, 0.05]], 300), r.multivariate_normal([0, 1.2], [[6, 0], [0, 0.05]], 300)])
truth = np.r_[np.zeros(300), np.ones(300)]
acc = lambda l: max(np.mean(l == truth), np.mean(l != truth))      # best of the two label matchings
klab = min((kmeans(Z, 2, s) for s in range(10)), key=lambda t: t[2])[0]
gR = gmm(Z, 2, klab, iters=200)[0]
print(f"k-means accuracy {acc(klab):.3f}    GMM accuracy {acc(gR.argmax(1)):.3f}")

In [ ]:
# @title Solution 5 (double-click to show the code)
r = np.random.default_rng(3)
Z = np.vstack([r.multivariate_normal([0, 0], [[6, 0], [0, 0.05]], 300), r.multivariate_normal([0, 1.2], [[6, 0], [0, 0.05]], 300)])
truth = np.r_[np.zeros(300), np.ones(300)]
acc = lambda l: max(np.mean(l == truth), np.mean(l != truth))      # best of the two label matchings
klab = min((kmeans(Z, 2, s) for s in range(10)), key=lambda t: t[2])[0]
gR = gmm(Z, 2, klab, iters=200)[0]
print(f"k-means accuracy {acc(klab):.3f}    GMM accuracy {acc(gR.argmax(1)):.3f}")

<details><summary><b>Answer and explanation</b></summary>

k-means gets 0.523, a coin flip, while the GMM gets 0.993. k-means' objective prefers to cut the two long strips across the middle (left half / right half), because that makes the round distances smaller. The GMM learns that each cluster is wide in x and thin in y, and separates them correctly. k-means is a GMM with equal, round covariances and hard assignments (Murphy §21.3), so it fails exactly when those assumptions fail.

Expected output:
```
k-means accuracy 0.523    GMM accuracy 0.993
```

</details>

## Milestone 6 · Choose K with BIC

For K = 1 … 6, fit a GMM on iris (initialised from k-means) and compute BIC = −2·log L + p·log N, where p counts every free parameter. Lower is better. Which K wins?

In [ ]:
# Your turn
N, D = X.shape
for k in range(1, 7):
    init = min((kmeans(X, k, s) for s in range(10)), key=lambda t: t[2])[0]
    ll = gmm(X, k, init)[4][-1]
    p = ...  # TODO: mixing weights + means + covariances
    print(f"K = {k}: log L {ll:8.1f}   params {p:3d}   BIC {-2 * ll + p * np.log(N):7.1f}")

In [ ]:
# @title Solution 6 (double-click to show the code)
N, D = X.shape
for k in range(1, 7):
    init = min((kmeans(X, k, s) for s in range(10)), key=lambda t: t[2])[0]
    ll = gmm(X, k, init)[4][-1]
    p = (k - 1) + k * D + k * D * (D + 1) // 2  # mixing weights + means + covariances
    print(f"K = {k}: log L {ll:8.1f}   params {p:3d}   BIC {-2 * ll + p * np.log(N):7.1f}")

<details><summary><b>Answer and explanation</b></summary>

BIC picks K = 2 (794.7), narrowly ahead of K = 3 (801.5). Going from 2 to 3 components raises the log-likelihood by 34 but costs 15 more parameters, and BIC charges log(150) ≈ 5 each. With only 150 points, BIC finds that versicolor and virginica aren't separated well enough to justify a whole extra full-covariance Gaussian. The 'true' answer of 3 species is not what the data alone can confidently support. BIC is a large-sample approximation to Bayesian model comparison (Murphy §5.2.5).

Expected output:
```
K = 1: log L   -490.3   params  14   BIC  1050.7
K = 2: log L   -324.7   params  29   BIC   794.7
K = 3: log L   -290.5   params  44   BIC   801.5
K = 4: log L   -275.0   params  59   BIC   845.7
K = 5: log L   -263.2   params  74   BIC   897.1
K = 6: log L   -245.2   params  89   BIC   936.3
```

</details>

## Milestone 7 · Soft assignments

Unlike k-means, a GMM says how sure it is. Using the K = 3 iris fit from step 4, count the points whose largest responsibility is below 0.9, and show which species they belong to.

In [ ]:
# Your turn
unsure = ...  # TODO: the model hedges on these points
print(f"{unsure.sum()} uncertain points; their species:", {str(iris.target_names[s]): int(np.sum(species[unsure] == s)) for s in range(3)})

In [ ]:
# @title Solution 7 (double-click to show the code)
unsure = R.max(1) < 0.9  # the model hedges on these points
print(f"{unsure.sum()} uncertain points; their species:", {str(iris.target_names[s]): int(np.sum(species[unsure] == s)) for s in range(3)})

<details><summary><b>Answer and explanation</b></summary>

Only 3 points are uncertain (largest responsibility below 0.9): 2 versicolor and 1 virginica, sitting where the two species overlap. Setosa is never in doubt. k-means would have hard-assigned these points without a word. A soft assignment tells you which labels to double-check, and EM uses exactly these fractional weights in its M-step.

Expected output:
```
3 uncertain points; their species: {'setosa': 0, 'versicolor': 2, 'virginica': 1}
```

</details>